# 99 — Build hierarchically integrated virtual T1 shot gathers (SAFE, v4)

Each virtual shot is assembled sequentially:

1. T1_1m establishes the preferred base gather.
2. T1_2m is correlated, aligned and normally super-stacked at coincident receivers.
3. Nodal data is correlated and aligned; by default it only fills unoccupied receivers.
4. T1_Streamer is correlated and aligned; by default it only fills unoccupied receivers.

Alignment is performed before output trimming and can search up to +/-0.5 s.
The common output duration is independent of the correlation window and is
configurable below.

## 1. Configuration

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import math

import numpy as np
import pandas as pd
from scipy.signal import hilbert, butter, sosfiltfilt

from obspy import read, Stream, Trace, UTCDateTime

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
PLAN_ROOT = PROJECT_ROOT / '98_virtual_T1_integration_plan'
OUT_ROOT = PROJECT_ROOT / '99_virtual_T1_shot_gathers'
MSEED_ROOT = OUT_ROOT / 'per_shot_mseed'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
MSEED_ROOT.mkdir(parents=True, exist_ok=True)

SHOT_CATALOG_PATH = PLAN_ROOT / '98_virtual_T1_shot_catalog.csv'
PRODUCT_PLAN_PATH = PLAN_ROOT / '98_virtual_T1_product_plan.csv'

COMPONENT = 'Z'
TARGET_SAMPLING_RATE_HZ = 1000.0

# Common final output interval. Longer secondary records are aligned in full
# before this interval is extracted. A T1_1m trace may contain NaNs/zeros after
# its approximately 0.4 s record ends; longer products retain later energy.
OUTPUT_START_S = -0.250
OUTPUT_END_S = 0.750

# Absolute time origin.
#
# Triggered T1_1m/T1_2m products remain authoritative when present. Airwave
# backprojection is applied only when every included product for a virtual shot
# is nodal. The fitted moveout is projected back to the known source position:
#
#     t_arrival(x) = t_shot + abs(x - x_source) / c_air
#
NODAL_ONLY_TIME_ORIGIN_MODE = 'airwave_auto_if_good'
# Other options: 'airwave_qc_only', 'none'.

SHOT_TIME_OVERRIDE_PATH = (
    PLAN_ROOT / '98_virtual_T1_shot_time_overrides.csv'
)

AIRWAVE_VELOCITY_MIN_MPS = 320.0
AIRWAVE_VELOCITY_MAX_MPS = 370.0
AIRWAVE_VELOCITY_STEP_MPS = 1.0
AIRWAVE_T0_MIN_S = -0.500
AIRWAVE_T0_MAX_S = 0.500
AIRWAVE_T0_STEP_S = 0.001

# Airwave-sensitive preprocessing.
AIRWAVE_FREQMIN_HZ = 30.0
AIRWAVE_FREQMAX_HZ = 180.0
AIRWAVE_ENVELOPE_SMOOTH_S = 0.006
AIRWAVE_MOVEOUT_SAMPLE_HALF_WIDTH_S = 0.004

# Use moderate/long offsets where acoustic and seismic moveout are easier to
# distinguish. These limits are relative to the known source position.
AIRWAVE_MIN_OFFSET_M = 20.0
AIRWAVE_MAX_OFFSET_M = 200.0
AIRWAVE_MIN_RECEIVERS = 6

# Acceptance and ambiguity tests.
AIRWAVE_MIN_SCORE = 2.0
AIRWAVE_MIN_PEAK_RATIO = 1.15
AIRWAVE_EXCLUSION_AROUND_BEST_S = 0.025
AIRWAVE_PICK_SEARCH_HALF_WIDTH_S = 0.020
AIRWAVE_MIN_TRACE_PEAK_Z = 1.0
AIRWAVE_MAX_RESIDUAL_MAD_S = 0.015
AIRWAVE_MAX_ABS_MEDIAN_RESIDUAL_S = 0.010



RECEIVER_MATCH_TOLERANCE_M = 0.25

# Requested hierarchy and default super-stack policy.
INTEGRATION_PRIORITY = {
    'T1_1m': 10,
    'T1_2m': 20,
    'nodal': 30,
    'T1_Streamer': 40,
}
T1_2m_SUPER_STACK = True
T1_NODAL_SUPER_STACK = False
T1_STREAMER_SUPER_STACK = False
SUPER_STACK_BY_ROLE = {
    'T1_1m': False,
    'T1_2m': T1_2m_SUPER_STACK,
    'nodal': T1_NODAL_SUPER_STACK,
    'T1_Streamer': T1_STREAMER_SUPER_STACK,
}

# Alignment is two-stage. Coarse alignment uses smoothed envelopes and can move
# a shorter 0.4 s reference within records as long as the configured search
# allows. Fine alignment uses the signed waveform close to the coarse solution.
MAX_COARSE_ALIGNMENT_SHIFT_S = 0.500
COARSE_ALIGNMENT_STEP_S = 0.005
MAX_FINE_ALIGNMENT_SHIFT_S = 0.020
FINE_ALIGNMENT_STEP_S = 0.001
ENVELOPE_SMOOTH_S = 0.008
MIN_ALIGNMENT_OVERLAP_S = 0.080
MIN_COMMON_RECEIVERS_FOR_ALIGNMENT = 2
ALIGNMENT_MIN_ENVELOPE_CORRELATION = 0.55
ALIGNMENT_MIN_WAVEFORM_CORRELATION = 0.35
SUPER_STACK_MIN_CORRELATION = 0.90
MAX_RECEIVER_LAG_MAD_S = 0.010
MIN_RECEIVER_LAG_AGREEMENT_FRACTION = 0.70
LAG_AGREEMENT_TOLERANCE_S = 0.010
MIN_CORRELATION_PEAK_RATIO = 1.10


# Products that cannot be aligned are not silently incorporated. Set this True
# only for an intentional diagnostic build using their initial plan shift.
ALLOW_UNALIGNED_NEW_RECEIVERS = False

# Nodal stack weight reflects accepted repeated shots. Original individual
# Geode/streamer gathers have unit weight.
NODAL_WEIGHT_COLUMN = 'n_accepted_members'
RAW_GEODE_WEIGHT = 1.0

VIRTUAL_EPOCH = UTCDateTime(2000, 1, 1)
SHOT_TIME_STRIDE_S = 10.0
WRITE_ALL_SHOTS_MSEED = True
MSEED_ENCODING = 'FLOAT32'
REQUIRE_GEODE_OUTPUT_IF_PLANNED = True

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Output:', OUT_ROOT)
print('Integration priority:', INTEGRATION_PRIORITY)
print('Super-stack policy:', SUPER_STACK_BY_ROLE)
print('Alignment search: +/-', MAX_COARSE_ALIGNMENT_SHIFT_S, 's')
print('Output relative window:', OUTPUT_START_S, 'to', OUTPUT_END_S, 's')

Output: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers
Integration priority: {'T1_1m': 10, 'T1_2m': 20, 'nodal': 30, 'T1_Streamer': 40}
Super-stack policy: {'T1_1m': False, 'T1_2m': True, 'nodal': False, 'T1_Streamer': False}
Alignment search: +/- 0.5 s
Output relative window: -0.25 to 0.75 s


## 2. Load integration plan

In [2]:
for path in [SHOT_CATALOG_PATH, PRODUCT_PLAN_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing notebook-98 output: {path}')

shots = pd.read_csv(SHOT_CATALOG_PATH, low_memory=False)
products = pd.read_csv(PRODUCT_PLAN_PATH, low_memory=False)

products['include_in_virtual_shot'] = products.include_in_virtual_shot.astype(str).str.lower().isin(['true', '1', 'yes'])
products['initial_time_shift_s'] = pd.to_numeric(
    products.get('initial_time_shift_s', products.get('time_shift_to_canonical_s', 0.0)),
    errors='coerce',
).fillna(0.0)
products['source_x_m'] = pd.to_numeric(products.source_x_m, errors='coerce')
products['integration_role'] = products.get('integration_role', 'nodal').fillna('nodal')
products['integration_priority'] = pd.to_numeric(
    products.get('integration_priority', products.integration_role.map(INTEGRATION_PRIORITY)),
    errors='coerce',
).fillna(products.integration_role.map(INTEGRATION_PRIORITY)).astype(int)

products = products.loc[
    products.include_in_virtual_shot
    & products.component.astype(str).str.upper().eq(COMPONENT)
].copy()

unknown_roles = sorted(set(products.integration_role) - set(INTEGRATION_PRIORITY))
if unknown_roles:
    raise ValueError(f'Unknown integration roles: {unknown_roles}')



shot_time_overrides = {}
if SHOT_TIME_OVERRIDE_PATH.exists():
    override_frame = pd.read_csv(
        SHOT_TIME_OVERRIDE_PATH,
        low_memory=False,
    )
    if len(override_frame):
        enabled = override_frame.get(
            'enabled',
            False,
        )
        enabled = enabled.astype(str).str.lower().isin(
            ['true', '1', 'yes']
        )
        override_frame = override_frame.loc[enabled].copy()
        override_frame['shot_time_from_file_start_s'] = pd.to_numeric(
            override_frame['shot_time_from_file_start_s'],
            errors='coerce',
        )
        override_frame = override_frame.loc[
            override_frame.shot_time_from_file_start_s.notna()
        ]
        shot_time_overrides = dict(zip(
            override_frame.virtual_shot_number.astype(int),
            override_frame.shot_time_from_file_start_s.astype(float),
        ))

print('Enabled shot-time overrides:', len(shot_time_overrides))

print('Virtual shots:', len(shots))
print('Included products:', len(products))
display(products.groupby(
    ['integration_role', 'product_kind', 'survey'], dropna=False
).size().reset_index(name='n_products'))

Enabled shot-time overrides: 0
Virtual shots: 159
Included products: 354


,integration_role,product_kind,survey,n_products
0,T1_1m,geode_raw,T1_1m_refraction,39
1,T1_2m,geode_raw,T1_2m_refraction,36
2,T1_Streamer,geode_raw,T1_streamer_masw,80
3,nodal,nodal_stack,T1_1m_refraction,39
4,nodal,nodal_stack,T1_2m_refraction,36
5,nodal,nodal_stack,T1_streamer_masw,80
6,nodal,nodal_stack,NaN,44


## 3. Waveform and geometry helpers

In [3]:
def normalize_component(value):
    text = str(value).strip().upper()
    return text[-1] if text and text[-1] in 'ZNE' else text


def as_bool(value):
    return str(value).strip().lower() in {'true', '1', 'yes'}


def trace_receiver_x_m(trace, *, raw_geode=False):
    candidates = [
        getattr(trace.stats, 'receiver_x_m', np.nan),
        getattr(trace.stats, 'distance', np.nan),
    ]
    seg2 = getattr(trace.stats, 'seg2', None)
    if seg2 is not None:
        keys = ['RECEIVER_LOCATION'] if raw_geode else [
            'RECEIVER_LOCATION', 'RECEIVER_STATION_NUMBER'
        ]
        for key in keys:
            try:
                candidates.append(seg2.get(key, np.nan))
            except Exception:
                pass
    for candidate in candidates:
        value = pd.to_numeric(candidate, errors='coerce')
        if pd.notna(value):
            return float(value)
    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def read_product_stream(product):
    path = Path(str(product.waveform_path))
    if not path.exists():
        raise FileNotFoundError(path)
    stream = read(str(path))
    selected = Stream(
        tr.copy() for tr in stream
        if product.product_kind == 'geode_raw'
        or normalize_component(tr.stats.channel) == COMPONENT
    )
    if product.product_kind == 'geode_raw':
        for tr in selected:
            tr.stats.channel = 'GHZ'
    return selected


FT_TO_M = 0.3048


def streamer_receiver_positions(source_x_m, *, reverse=False):
    offsets_ft = 30.0 + 5.0 * np.arange(24, dtype=float)
    positions = float(source_x_m) - offsets_ft * FT_TO_M
    return positions[::-1] if reverse else positions


def assign_receiver_positions(stream, product):
    role = str(getattr(product, 'integration_role', ''))
    reverse = as_bool(getattr(product, 'reverse_trace_order_fallback', False))

    # Authoritative source-relative streamer geometry.
    if role == 'T1_Streamer':
        expected = int(pd.to_numeric(
            getattr(product, 'receiver_count_expected', 24),
            errors='coerce',
        ))
        if len(stream) != expected:
            raise ValueError(
                f'T1 streamer expected {expected} traces but read {len(stream)}'
            )
        positions = streamer_receiver_positions(
            float(product.source_x_m),
            reverse=reverse,
        )
        return [
            (float(receiver_x), stream[index].copy(), index)
            for index, receiver_x in enumerate(positions)
        ]

    output = []
    fallback_first = pd.to_numeric(
        getattr(product, 'receiver_first_x_m_fallback', np.nan),
        errors='coerce',
    )
    fallback_dx = pd.to_numeric(
        getattr(product, 'receiver_dx_m_fallback', np.nan),
        errors='coerce',
    )
    indices = list(range(len(stream)))
    if reverse:
        indices.reverse()

    for output_index, original_index in enumerate(indices):
        trace = stream[original_index].copy()
        receiver_x = trace_receiver_x_m(
            trace,
            raw_geode=(product.product_kind == 'geode_raw'),
        )
        if product.product_kind == 'geode_raw' and (
            not np.isfinite(receiver_x)
            or receiver_x < -1000
            or receiver_x > 10000
        ):
            if pd.isna(fallback_first) or pd.isna(fallback_dx):
                continue
            receiver_x = float(fallback_first + output_index * fallback_dx)
        if np.isfinite(receiver_x):
            output.append((float(receiver_x), trace, original_index))
    return output


def product_weight(product):
    if product.product_kind == 'nodal_stack':
        value = pd.to_numeric(getattr(product, NODAL_WEIGHT_COLUMN, np.nan), errors='coerce')
        return float(value) if pd.notna(value) and value > 0 else 1.0
    return RAW_GEODE_WEIGHT


def prepare_trace(trace):
    working = trace.copy()
    working.detrend('demean')
    target_rate = float(TARGET_SAMPLING_RATE_HZ)
    if not np.isclose(working.stats.sampling_rate, target_rate):
        working.interpolate(sampling_rate=target_rate, method='lanczos', a=12)
    data = np.asarray(working.data, dtype=float)
    times = np.arange(data.size, dtype=float) / target_rate
    return times, data


def sample_shifted(input_times, input_data, output_times, shift_s):
    # Positive shift moves the waveform later: output(t)=input(t-shift).
    return np.interp(
        output_times - float(shift_s), input_times, input_data,
        left=np.nan, right=np.nan,
    )


def smooth_envelope(data):
    finite = np.isfinite(data)
    filled = np.where(finite, data, 0.0)
    envelope = np.abs(hilbert(filled))
    n = max(1, int(round(ENVELOPE_SMOOTH_S * TARGET_SAMPLING_RATE_HZ)))
    if n > 1:
        envelope = np.convolve(envelope, np.ones(n) / n, mode='same')
    envelope[~finite] = np.nan
    return envelope


def normalized_correlation(a, b):
    valid = np.isfinite(a) & np.isfinite(b)
    minimum = max(3, int(round(MIN_ALIGNMENT_OVERLAP_S * TARGET_SAMPLING_RATE_HZ)))
    if valid.sum() < minimum:
        return np.nan
    x = np.asarray(a[valid], dtype=float)
    y = np.asarray(b[valid], dtype=float)
    x -= x.mean(); y -= y.mean()
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denom) if denom > 0 else np.nan


def receiver_cluster_key(receiver_x_m):
    return int(round(float(receiver_x_m) / RECEIVER_MATCH_TOLERANCE_M))



def robust_standardize(data):
    data = np.asarray(data, dtype=float)
    med = np.nanmedian(data)
    mad = 1.4826 * np.nanmedian(np.abs(data - med))
    if not np.isfinite(mad) or mad <= 0:
        mad = np.nanstd(data)
    if not np.isfinite(mad) or mad <= 0:
        mad = 1.0
    return (data - med) / mad


def airwave_characteristic(input_data):
    """Bandpass, envelope, smooth, and robustly standardize one trace."""
    data = np.asarray(input_data, dtype=float)
    finite = np.isfinite(data)
    filled = np.where(finite, data, 0.0)
    filled = filled - np.nanmedian(filled)

    nyquist = 0.5 * TARGET_SAMPLING_RATE_HZ
    low = max(0.001, AIRWAVE_FREQMIN_HZ / nyquist)
    high = min(0.999, AIRWAVE_FREQMAX_HZ / nyquist)
    if not 0 < low < high < 1:
        raise ValueError(
            'Invalid airwave bandpass for sampling rate '
            f'{TARGET_SAMPLING_RATE_HZ}: '
            f'{AIRWAVE_FREQMIN_HZ}-{AIRWAVE_FREQMAX_HZ} Hz'
        )

    sos = butter(
        4,
        [low, high],
        btype='bandpass',
        output='sos',
    )
    try:
        filtered = sosfiltfilt(sos, filled)
    except ValueError:
        # Very short traces may not satisfy filtfilt padding requirements.
        filtered = filled

    envelope = np.abs(hilbert(filtered))
    smooth_n = max(
        1,
        int(round(
            AIRWAVE_ENVELOPE_SMOOTH_S
            * TARGET_SAMPLING_RATE_HZ
        )),
    )
    if smooth_n > 1:
        envelope = np.convolve(
            envelope,
            np.ones(smooth_n, dtype=float) / smooth_n,
            mode='same',
        )
    envelope[~finite] = np.nan
    return robust_standardize(envelope)


def _windowed_interp_max(times, values, centers, half_width_s):
    """Maximum characteristic value near each predicted arrival time."""
    centers = np.asarray(centers, dtype=float)
    if half_width_s <= 0:
        return np.interp(
            centers,
            times,
            values,
            left=np.nan,
            right=np.nan,
        )

    offsets = np.arange(
        -half_width_s,
        half_width_s + 0.5 / TARGET_SAMPLING_RATE_HZ,
        1.0 / TARGET_SAMPLING_RATE_HZ,
    )
    sampled = np.vstack([
        np.interp(
            centers + offset,
            times,
            values,
            left=np.nan,
            right=np.nan,
        )
        for offset in offsets
    ])
    with np.errstate(all='ignore'):
        return np.nanmax(sampled, axis=0)


def _empty_airwave_result(status, n_receivers=0):
    return {
        'accepted': False,
        'status': status,
        't0_s': np.nan,
        'velocity_mps': np.nan,
        'score': np.nan,
        'second_score': np.nan,
        'peak_ratio': np.nan,
        'second_t0_s': np.nan,
        'second_velocity_mps': np.nan,
        'n_receivers': int(n_receivers),
        'n_supporting_receivers': 0,
        'receiver_residual_median_s': np.nan,
        'receiver_residual_mad_s': np.nan,
        'receiver_residual_max_abs_s': np.nan,
        'offset_min_m': np.nan,
        'offset_max_m': np.nan,
    }


def airwave_time_origin_fit(candidate_traces, source_x_m):
    """
    Estimate physical shot time in an untriggered nodal record.

    The airwave characteristic is sampled along trial moveouts and projected
    back to the known source coordinate. The fitted ``t0_s`` is therefore the
    predicted airwave time at the source, not the first arrival seen anywhere
    in the receiver array.
    """
    eligible = []
    for item in candidate_traces:
        offset_m = abs(
            float(item['receiver_x_m']) - float(source_x_m)
        )
        if not (
            AIRWAVE_MIN_OFFSET_M
            <= offset_m
            <= AIRWAVE_MAX_OFFSET_M
        ):
            continue
        if len(item['input_data']) < 8:
            continue
        eligible.append({
            **item,
            'offset_m': float(offset_m),
            'characteristic': airwave_characteristic(
                item['input_data']
            ),
        })

    if len(eligible) < AIRWAVE_MIN_RECEIVERS:
        result = _empty_airwave_result(
            'too_few_eligible_receivers',
            len(eligible),
        )
        if eligible:
            offsets = [item['offset_m'] for item in eligible]
            result['offset_min_m'] = float(min(offsets))
            result['offset_max_m'] = float(max(offsets))
        return result

    # All products are resampled to the target rate, but lengths can differ.
    # Each trace is interpolated on its own time vector.
    velocities = np.arange(
        AIRWAVE_VELOCITY_MIN_MPS,
        AIRWAVE_VELOCITY_MAX_MPS
        + AIRWAVE_VELOCITY_STEP_MPS / 2,
        AIRWAVE_VELOCITY_STEP_MPS,
    )
    t0_values = np.arange(
        AIRWAVE_T0_MIN_S,
        AIRWAVE_T0_MAX_S
        + AIRWAVE_T0_STEP_S / 2,
        AIRWAVE_T0_STEP_S,
    )

    score_grid = np.full(
        (len(velocities), len(t0_values)),
        np.nan,
        dtype=float,
    )
    support_grid = np.zeros_like(score_grid, dtype=int)

    for velocity_index, velocity in enumerate(velocities):
        per_receiver = []
        for item in eligible:
            predicted = (
                t0_values
                + item['offset_m'] / float(velocity)
            )
            sampled = _windowed_interp_max(
                item['input_times'],
                item['characteristic'],
                predicted,
                AIRWAVE_MOVEOUT_SAMPLE_HALF_WIDTH_S,
            )
            per_receiver.append(sampled)

        values = np.vstack(per_receiver)
        support = np.sum(np.isfinite(values), axis=0)
        with np.errstate(all='ignore'):
            scores = np.nanmedian(values, axis=0)
        scores[support < AIRWAVE_MIN_RECEIVERS] = np.nan
        score_grid[velocity_index] = scores
        support_grid[velocity_index] = support

    if not np.isfinite(score_grid).any():
        return _empty_airwave_result(
            'no_valid_backprojection_grid_points',
            len(eligible),
        )

    best_flat = int(np.nanargmax(score_grid))
    best_velocity_index, best_t0_index = np.unravel_index(
        best_flat,
        score_grid.shape,
    )
    best_score = float(score_grid[
        best_velocity_index,
        best_t0_index,
    ])
    best_velocity = float(velocities[best_velocity_index])
    best_t0 = float(t0_values[best_t0_index])
    n_used = int(support_grid[
        best_velocity_index,
        best_t0_index,
    ])

    # Find a genuinely separate competing source-time peak. Velocity variants
    # at the same source time are not counted as separate events.
    competing = score_grid.copy()
    t0_separation_mask = (
        np.abs(t0_values - best_t0)
        < AIRWAVE_EXCLUSION_AROUND_BEST_S
    )
    competing[:, t0_separation_mask] = np.nan

    if np.isfinite(competing).any():
        second_flat = int(np.nanargmax(competing))
        second_velocity_index, second_t0_index = np.unravel_index(
            second_flat,
            competing.shape,
        )
        second_score = float(competing[
            second_velocity_index,
            second_t0_index,
        ])
        second_t0 = float(t0_values[second_t0_index])
        second_velocity = float(
            velocities[second_velocity_index]
        )
    else:
        second_score = np.nan
        second_t0 = np.nan
        second_velocity = np.nan

    peak_ratio = (
        best_score / second_score
        if np.isfinite(second_score) and second_score > 0
        else np.inf
    )

    # Receiver-level residual QC. Pick the local characteristic maximum near
    # the predicted airwave arrival on each receiver.
    residuals = []
    supporting_scores = []
    dt = 1.0 / TARGET_SAMPLING_RATE_HZ
    for item in eligible:
        predicted = best_t0 + item['offset_m'] / best_velocity
        times = item['input_times']
        characteristic = item['characteristic']
        mask = (
            (times >= predicted - AIRWAVE_PICK_SEARCH_HALF_WIDTH_S)
            & (times <= predicted + AIRWAVE_PICK_SEARCH_HALF_WIDTH_S)
            & np.isfinite(characteristic)
        )
        if not np.any(mask):
            continue
        local_indices = np.flatnonzero(mask)
        peak_local = int(np.nanargmax(
            characteristic[local_indices]
        ))
        peak_index = int(local_indices[peak_local])
        peak_score = float(characteristic[peak_index])
        if peak_score < AIRWAVE_MIN_TRACE_PEAK_Z:
            continue
        picked_time = float(times[peak_index])
        residuals.append(picked_time - predicted)
        supporting_scores.append(peak_score)

    if residuals:
        residuals = np.asarray(residuals, dtype=float)
        residual_median = float(np.nanmedian(residuals))
        residual_mad = float(
            1.4826
            * np.nanmedian(
                np.abs(residuals - residual_median)
            )
        )
        residual_max_abs = float(
            np.nanmax(np.abs(residuals))
        )
    else:
        residual_median = np.nan
        residual_mad = np.nan
        residual_max_abs = np.nan

    n_supporting = len(residuals)
    accepted = (
        best_score >= AIRWAVE_MIN_SCORE
        and peak_ratio >= AIRWAVE_MIN_PEAK_RATIO
        and n_supporting >= AIRWAVE_MIN_RECEIVERS
        and np.isfinite(residual_mad)
        and residual_mad <= AIRWAVE_MAX_RESIDUAL_MAD_S
        and abs(residual_median)
        <= AIRWAVE_MAX_ABS_MEDIAN_RESIDUAL_S
    )

    if accepted:
        status = 'accepted'
    elif best_score < AIRWAVE_MIN_SCORE:
        status = 'low_backprojection_score'
    elif peak_ratio < AIRWAVE_MIN_PEAK_RATIO:
        status = 'ambiguous_multiple_airwave_peaks'
    elif n_supporting < AIRWAVE_MIN_RECEIVERS:
        status = 'too_few_receiver_picks'
    elif not np.isfinite(residual_mad):
        status = 'no_receiver_residual_solution'
    elif residual_mad > AIRWAVE_MAX_RESIDUAL_MAD_S:
        status = 'inconsistent_receiver_residuals'
    else:
        status = 'biased_receiver_residuals'

    offsets = [item['offset_m'] for item in eligible]
    return {
        'accepted': bool(accepted),
        'status': status,
        't0_s': best_t0,
        'velocity_mps': best_velocity,
        'score': best_score,
        'second_score': second_score,
        'peak_ratio': float(peak_ratio),
        'second_t0_s': second_t0,
        'second_velocity_mps': second_velocity,
        'n_receivers': int(n_used),
        'n_supporting_receivers': int(n_supporting),
        'receiver_residual_median_s': residual_median,
        'receiver_residual_mad_s': residual_mad,
        'receiver_residual_max_abs_s': residual_max_abs,
        'offset_min_m': float(min(offsets)),
        'offset_max_m': float(max(offsets)),
    }


def best_shift_for_pair(reference_data, candidate_times, candidate_data, initial_shift_s):
    reference_times = OUTPUT_TIMES
    reference_env = smooth_envelope(reference_data)
    candidate_env = smooth_envelope(candidate_data)

    coarse_offsets = np.arange(
        -MAX_COARSE_ALIGNMENT_SHIFT_S,
        MAX_COARSE_ALIGNMENT_SHIFT_S + COARSE_ALIGNMENT_STEP_S / 2,
        COARSE_ALIGNMENT_STEP_S,
    )
    coarse_scores = []
    for offset in coarse_offsets:
        shift = float(initial_shift_s + offset)
        sampled = sample_shifted(candidate_times, candidate_env, reference_times, shift)
        coarse_scores.append(normalized_correlation(reference_env, sampled))
    coarse_scores = np.asarray(coarse_scores, dtype=float)
    if not np.isfinite(coarse_scores).any():
        return None
    coarse_index = int(np.nanargmax(coarse_scores))
    coarse_shift = float(initial_shift_s + coarse_offsets[coarse_index])
    coarse_corr = float(coarse_scores[coarse_index])
    finite_coarse = coarse_scores[np.isfinite(coarse_scores)]
    if finite_coarse.size > 1:
        ordered = np.sort(finite_coarse)[::-1]
        coarse_peak_ratio = (
            float(ordered[0] / ordered[1])
            if ordered[1] > 0 else np.inf
        )
    else:
        coarse_peak_ratio = np.nan

    fine_offsets = np.arange(
        -MAX_FINE_ALIGNMENT_SHIFT_S,
        MAX_FINE_ALIGNMENT_SHIFT_S + FINE_ALIGNMENT_STEP_S / 2,
        FINE_ALIGNMENT_STEP_S,
    )
    fine_scores = []
    for offset in fine_offsets:
        shift = coarse_shift + float(offset)
        sampled = sample_shifted(candidate_times, candidate_data, reference_times, shift)
        fine_scores.append(normalized_correlation(reference_data, sampled))
    fine_scores = np.asarray(fine_scores, dtype=float)
    if np.isfinite(fine_scores).any():
        fine_index = int(np.nanargmax(fine_scores))
        fine_shift = float(coarse_shift + fine_offsets[fine_index])
        fine_corr = float(fine_scores[fine_index])
    else:
        fine_shift = coarse_shift
        fine_corr = np.nan
    return {
        'coarse_shift_s': coarse_shift,
        'coarse_envelope_corr': coarse_corr,
        'fine_shift_s': fine_shift,
        'fine_waveform_corr': fine_corr,
        'coarse_peak_ratio': coarse_peak_ratio,
    }


def estimate_product_alignment(gather, candidate_traces, initial_shift_s):
    pair_results = []
    by_key = {receiver_cluster_key(item['receiver_x_m']): item for item in candidate_traces}
    for key, existing in gather.items():
        candidate = by_key.get(key)
        if candidate is None:
            continue
        result = best_shift_for_pair(
            existing['data'], candidate['input_times'], candidate['input_data'], initial_shift_s
        )
        if result is not None:
            result.update({
                'receiver_cluster_key': key,
                'existing_receiver_x_m': existing['receiver_x_m'],
                'candidate_receiver_x_m': candidate['receiver_x_m'],
            })
            pair_results.append(result)

    if not pair_results:
        return {
            'accepted': False, 'status': 'no_common_receivers',
            'n_common_receivers': 0, 'applied_shift_s': np.nan,
            'median_envelope_corr': np.nan, 'median_waveform_corr': np.nan,
            'pair_results': [],
        }

    frame = pd.DataFrame(pair_results)
    applied_shift = float(np.nanmedian(frame.fine_shift_s))
    median_env = float(np.nanmedian(frame.coarse_envelope_corr))
    median_wave = float(np.nanmedian(frame.fine_waveform_corr))
    lag_mad = float(
        1.4826 * np.nanmedian(np.abs(frame.fine_shift_s - applied_shift))
    )
    lag_agreement_fraction = float(np.nanmean(
        np.abs(frame.fine_shift_s - applied_shift)
        <= LAG_AGREEMENT_TOLERANCE_S
    ))
    median_peak_ratio = float(np.nanmedian(frame.coarse_peak_ratio))
    enough = len(frame) >= MIN_COMMON_RECEIVERS_FOR_ALIGNMENT
    correlation_ok = (
        median_env >= ALIGNMENT_MIN_ENVELOPE_CORRELATION
        and (
            np.isnan(median_wave)
            or median_wave >= ALIGNMENT_MIN_WAVEFORM_CORRELATION
        )
    )
    lag_consistent = (
        lag_mad <= MAX_RECEIVER_LAG_MAD_S
        and lag_agreement_fraction >= MIN_RECEIVER_LAG_AGREEMENT_FRACTION
    )
    unique_peak = (
        np.isnan(median_peak_ratio)
        or median_peak_ratio >= MIN_CORRELATION_PEAK_RATIO
    )
    accepted = enough and correlation_ok and lag_consistent and unique_peak

    if accepted:
        status = 'accepted'
    elif not enough:
        status = 'insufficient_common_receivers'
    elif not correlation_ok:
        status = 'low_correlation'
    elif not lag_consistent:
        status = 'inconsistent_receiver_lags'
    else:
        status = 'ambiguous_correlation_peak'

    return {
        'accepted': bool(accepted),
        'status': status,
        'n_common_receivers': len(frame),
        'applied_shift_s': applied_shift,
        'median_envelope_corr': median_env,
        'median_waveform_corr': median_wave,
        'receiver_lag_mad_s': lag_mad,
        'receiver_lag_agreement_fraction': lag_agreement_fraction,
        'median_correlation_peak_ratio': median_peak_ratio,
        'pair_results': pair_results,
    }


def valid_weighted_stack(existing_data, candidate_data, existing_weight, candidate_weight):
    arrays = np.vstack([existing_data, candidate_data])
    weights = np.asarray([existing_weight, candidate_weight], dtype=float)
    finite = np.isfinite(arrays)
    numerator = np.where(finite, arrays * weights[:, None], 0.0).sum(axis=0)
    denominator = np.where(finite, weights[:, None], 0.0).sum(axis=0)
    return np.divide(numerator, denominator, out=np.full(arrays.shape[1], np.nan), where=denominator > 0)


def short_station_code(receiver_x_m):
    return f'R{int(round(receiver_x_m * 10)):04d}'[:5]


OUTPUT_TIMES = np.arange(
    OUTPUT_START_S,
    OUTPUT_END_S,
    1.0 / TARGET_SAMPLING_RATE_HZ,
)

## 4. Build every virtual shot

In [4]:
shot_rows = []
trace_rows = []
contribution_rows = []
alignment_rows = []
alignment_pair_rows = []
product_error_rows = []
geometry_qc_rows = []
shot_time_qc_rows = []
multiple_event_qc_rows = []
all_shot_stream = Stream()

for shot in shots.sort_values('virtual_shot_number').itertuples(index=False):
    shot_products = products.loc[
        products.virtual_shot_number.eq(shot.virtual_shot_number)
    ].sort_values(['integration_priority', 'product_id'], kind='stable')

    shot_roles = set(
        shot_products.integration_role.astype(str)
    )
    nodal_only_shot = (
        len(shot_roles) > 0
        and shot_roles == {'nodal'}
    )
    has_triggered_reference = bool(
        shot_roles.intersection({'T1_1m', 'T1_2m'})
    )
    enabled_manual_override = shot_time_overrides.get(
        int(shot.virtual_shot_number)
    )
    shot_timing_method = None
    shot_timing_status = None
    shot_estimated_t0_s = np.nan
    shot_applied_base_shift_s = np.nan

    gather = {}  # one authoritative output trace per receiver-position cluster
    product_errors = []
    n_loaded_candidate_traces = 0

    for product in shot_products.itertuples(index=False):
        try:
            stream = read_product_stream(product)
            positioned = assign_receiver_positions(stream, product)
            weight = product_weight(product)
            candidate_traces = []
            for receiver_x, trace, original_trace_index in positioned:
                input_times, input_data = prepare_trace(trace)
                candidate_traces.append({
                    'receiver_x_m': receiver_x,
                    'cluster_key': receiver_cluster_key(receiver_x),
                    'input_times': input_times,
                    'input_data': input_data,
                    'original_trace_index': original_trace_index,
                })
            n_loaded_candidate_traces += len(candidate_traces)

            role = str(product.integration_role)
            receiver_values = np.asarray(
                [item['receiver_x_m'] for item in candidate_traces],
                dtype=float,
            )
            geometry_qc_rows.append({
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'product_id': product.product_id,
                'integration_role': role,
                'n_receivers': len(receiver_values),
                'receiver_x_min_m': float(np.nanmin(receiver_values)) if len(receiver_values) else np.nan,
                'receiver_x_max_m': float(np.nanmax(receiver_values)) if len(receiver_values) else np.nan,
                'minimum_spacing_m': float(np.nanmin(np.diff(np.sort(np.unique(receiver_values))))) if len(np.unique(receiver_values)) > 1 else np.nan,
                'n_exact_duplicate_positions': int(len(receiver_values) - len(np.unique(np.round(receiver_values, 6)))),
                'geometry_mode': getattr(product, 'receiver_geometry_mode', None),
            })

            initial_shift = float(product.initial_time_shift_s)
            super_stack_enabled = bool(SUPER_STACK_BY_ROLE[role])

            if not gather:
                # Absolute timing is handled independently from relative
                # product alignment.
                #
                # 1. Explicit reviewed override, when enabled.
                # 2. Nodal-only airwave backprojection.
                # 3. Preserve triggered T1_1m/T1_2m timing.
                # 4. Otherwise retain the plan's initial shift and flag the
                #    absolute time as unresolved.
                if enabled_manual_override is not None:
                    airwave_fit = _empty_airwave_result(
                        'manual_override_not_evaluated',
                        len(candidate_traces),
                    )
                    base_shift = -float(enabled_manual_override)
                    timing_method = 'manual_override'
                    timing_status = 'accepted_manual_override'
                    fit_applied = False
                    estimated_t0_s = float(
                        enabled_manual_override
                    )
                elif nodal_only_shot and role == 'nodal':
                    airwave_fit = airwave_time_origin_fit(
                        candidate_traces,
                        float(shot.source_x_m),
                    )
                    fit_applied = (
                        NODAL_ONLY_TIME_ORIGIN_MODE
                        == 'airwave_auto_if_good'
                        and airwave_fit['accepted']
                    )
                    if fit_applied:
                        base_shift = -float(
                            airwave_fit['t0_s']
                        )
                        timing_method = (
                            'nodal_airwave_backprojection'
                        )
                        timing_status = 'accepted'
                    else:
                        base_shift = initial_shift
                        timing_method = (
                            'nodal_airwave_backprojection_qc'
                        )
                        timing_status = airwave_fit['status']
                    estimated_t0_s = airwave_fit['t0_s']
                elif has_triggered_reference and role in {
                    'T1_1m',
                    'T1_2m',
                }:
                    airwave_fit = _empty_airwave_result(
                        'triggered_reference_not_evaluated',
                        len(candidate_traces),
                    )
                    fit_applied = False
                    base_shift = initial_shift
                    timing_method = 'triggered_reference'
                    timing_status = (
                        'triggered_timing_preserved'
                    )
                    estimated_t0_s = np.nan
                else:
                    airwave_fit = _empty_airwave_result(
                        'absolute_time_unresolved',
                        len(candidate_traces),
                    )
                    fit_applied = False
                    base_shift = initial_shift
                    timing_method = (
                        'initial_plan_shift_unresolved'
                    )
                    timing_status = (
                        'absolute_time_unresolved'
                    )
                    estimated_t0_s = np.nan

                shot_timing_method = timing_method
                shot_timing_status = timing_status
                shot_estimated_t0_s = estimated_t0_s
                shot_applied_base_shift_s = base_shift

                shot_time_qc_rows.append({
                    'virtual_shot_number': int(
                        shot.virtual_shot_number
                    ),
                    'virtual_shot_id': shot.virtual_shot_id,
                    'source_x_m': float(shot.source_x_m),
                    'base_product_id': product.product_id,
                    'base_integration_role': role,
                    'shot_roles': ' | '.join(sorted(shot_roles)),
                    'nodal_only_shot': nodal_only_shot,
                    'has_triggered_reference': (
                        has_triggered_reference
                    ),
                    'timing_method': timing_method,
                    'timing_status': timing_status,
                    'mode': NODAL_ONLY_TIME_ORIGIN_MODE,
                    'fit_status': airwave_fit['status'],
                    'fit_accepted': airwave_fit['accepted'],
                    'fit_applied': fit_applied,
                    'estimated_shot_time_from_file_start_s': (
                        estimated_t0_s
                    ),
                    'applied_base_shift_s': base_shift,
                    'airwave_velocity_mps': (
                        airwave_fit['velocity_mps']
                    ),
                    'airwave_score': airwave_fit['score'],
                    'airwave_second_score': (
                        airwave_fit['second_score']
                    ),
                    'airwave_peak_ratio': (
                        airwave_fit['peak_ratio']
                    ),
                    'second_airwave_t0_s': (
                        airwave_fit['second_t0_s']
                    ),
                    'second_airwave_velocity_mps': (
                        airwave_fit['second_velocity_mps']
                    ),
                    'n_receivers_used': (
                        airwave_fit['n_receivers']
                    ),
                    'n_supporting_receivers': (
                        airwave_fit[
                            'n_supporting_receivers'
                        ]
                    ),
                    'receiver_residual_median_s': (
                        airwave_fit[
                            'receiver_residual_median_s'
                        ]
                    ),
                    'receiver_residual_mad_s': (
                        airwave_fit[
                            'receiver_residual_mad_s'
                        ]
                    ),
                    'receiver_residual_max_abs_s': (
                        airwave_fit[
                            'receiver_residual_max_abs_s'
                        ]
                    ),
                    'offset_min_m': (
                        airwave_fit['offset_min_m']
                    ),
                    'offset_max_m': (
                        airwave_fit['offset_max_m']
                    ),
                })

                if (
                    nodal_only_shot
                    and airwave_fit['status']
                    == 'ambiguous_multiple_airwave_peaks'
                ):
                    multiple_event_qc_rows.append({
                        'virtual_shot_number': int(
                            shot.virtual_shot_number
                        ),
                        'virtual_shot_id': (
                            shot.virtual_shot_id
                        ),
                        'source_x_m': float(
                            shot.source_x_m
                        ),
                        'product_id': product.product_id,
                        'integration_role': role,
                        'flag': (
                            'multiple_coherent_airwave_peaks'
                        ),
                        'best_t0_s': (
                            airwave_fit['t0_s']
                        ),
                        'second_t0_s': (
                            airwave_fit['second_t0_s']
                        ),
                        'best_velocity_mps': (
                            airwave_fit['velocity_mps']
                        ),
                        'second_velocity_mps': (
                            airwave_fit[
                                'second_velocity_mps'
                            ]
                        ),
                        'peak_ratio': (
                            airwave_fit['peak_ratio']
                        ),
                    })

                alignment = {
                    'accepted': True,
                    'status': (
                        'base_product_' + timing_status
                    ),
                    'n_common_receivers': 0,
                    'applied_shift_s': base_shift,
                    'median_envelope_corr': np.nan,
                    'median_waveform_corr': np.nan,
                    'receiver_lag_mad_s': np.nan,
                    'receiver_lag_agreement_fraction': np.nan,
                    'median_correlation_peak_ratio': np.nan,
                    'pair_results': [],
                }
            else:
                alignment = estimate_product_alignment(gather, candidate_traces, initial_shift)

            alignment_rows.append({
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'product_id': product.product_id,
                'integration_role': role,
                'integration_priority': int(product.integration_priority),
                'super_stack_enabled': super_stack_enabled,
                'initial_shift_s': initial_shift,
                'alignment_status': alignment['status'],
                'alignment_accepted': alignment['accepted'],
                'n_common_receivers': alignment['n_common_receivers'],
                'applied_shift_s': alignment['applied_shift_s'],
                'median_envelope_corr': alignment['median_envelope_corr'],
                'median_waveform_corr': alignment['median_waveform_corr'],
                'receiver_lag_mad_s': alignment.get('receiver_lag_mad_s', np.nan),
                'receiver_lag_agreement_fraction': alignment.get(
                    'receiver_lag_agreement_fraction', np.nan
                ),
                'median_correlation_peak_ratio': alignment.get(
                    'median_correlation_peak_ratio', np.nan
                ),
            })
            if alignment['status'] in {
                'inconsistent_receiver_lags',
                'ambiguous_correlation_peak',
            }:
                multiple_event_qc_rows.append({
                    'virtual_shot_number': int(shot.virtual_shot_number),
                    'virtual_shot_id': shot.virtual_shot_id,
                    'source_x_m': float(shot.source_x_m),
                    'product_id': product.product_id,
                    'integration_role': role,
                    'reason': alignment['status'],
                    'receiver_lag_mad_s': alignment.get('receiver_lag_mad_s', np.nan),
                    'receiver_lag_agreement_fraction': alignment.get(
                        'receiver_lag_agreement_fraction', np.nan
                    ),
                    'median_correlation_peak_ratio': alignment.get(
                        'median_correlation_peak_ratio', np.nan
                    ),
                })
            for pair in alignment['pair_results']:
                alignment_pair_rows.append({
                    'virtual_shot_number': int(shot.virtual_shot_number),
                    'product_id': product.product_id,
                    'integration_role': role,
                    **pair,
                })

            if not alignment['accepted'] and not ALLOW_UNALIGNED_NEW_RECEIVERS:
                continue
            applied_shift = (
                float(alignment['applied_shift_s'])
                if np.isfinite(alignment['applied_shift_s'])
                else initial_shift
            )

            pair_corr_by_key = {
                int(pair['receiver_cluster_key']): pair['fine_waveform_corr']
                for pair in alignment['pair_results']
            }

            for candidate in candidate_traces:
                key = candidate['cluster_key']
                shifted = sample_shifted(
                    candidate['input_times'], candidate['input_data'], OUTPUT_TIMES, applied_shift
                )
                contribution = {
                    'product_id': product.product_id,
                    'product_kind': product.product_kind,
                    'survey': product.survey,
                    'integration_role': role,
                    'waveform_path': product.waveform_path,
                    'input_receiver_x_m': candidate['receiver_x_m'],
                    'original_trace_index': candidate['original_trace_index'],
                    'applied_shift_s': applied_shift,
                    'weight': weight,
                }

                if key not in gather:
                    gather[key] = {
                        'receiver_x_m': candidate['receiver_x_m'],
                        'data': shifted,
                        'weight': weight,
                        'primary_role': role,
                        'primary_family': product.receiver_family,
                        'contributions': [contribution],
                    }
                    continue

                # Occupied location: lower-priority trace is ignored unless its
                # role permits super-stacking and its receiver-level waveform
                # correlation is exceptionally good.
                receiver_corr = pair_corr_by_key.get(key, np.nan)
                if super_stack_enabled and np.isfinite(receiver_corr) and receiver_corr >= SUPER_STACK_MIN_CORRELATION:
                    existing = gather[key]
                    existing['data'] = valid_weighted_stack(
                        existing['data'], shifted, existing['weight'], weight
                    )
                    existing['weight'] += weight
                    contribution['super_stacked'] = True
                    contribution['receiver_waveform_corr'] = receiver_corr
                    existing['contributions'].append(contribution)
                else:
                    # Explicitly record why a coincident lower-priority trace was not used.
                    contribution_rows.append({
                        'virtual_shot_number': int(shot.virtual_shot_number),
                        'virtual_shot_id': shot.virtual_shot_id,
                        'source_x_m': float(shot.source_x_m),
                        'output_receiver_x_m': gather[key]['receiver_x_m'],
                        **contribution,
                        'used_in_output': False,
                        'super_stacked': False,
                        'receiver_waveform_corr': receiver_corr,
                        'decision': 'occupied_super_stack_disabled_or_below_threshold',
                    })

        except Exception as exc:
            error_row = {
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'product_id': product.product_id,
                'product_kind': product.product_kind,
                'integration_role': getattr(product, 'integration_role', None),
                'receiver_family': product.receiver_family,
                'survey': product.survey,
                'waveform_path': product.waveform_path,
                'error': repr(exc),
            }
            product_errors.append(error_row)
            product_error_rows.append(error_row)

    output_stream = Stream()
    for trace_number, (key, entry) in enumerate(sorted(gather.items(), key=lambda item: item[1]['receiver_x_m']), start=1):
        receiver_x = float(entry['receiver_x_m'])
        data = np.nan_to_num(entry['data'], nan=0.0).astype(np.float32)
        trace = Trace(data=data)
        trace.stats.network = 'VT'
        trace.stats.station = short_station_code(receiver_x)
        role_code = {'T1_1m':'1M', 'T1_2m':'2M', 'nodal':'ND', 'T1_Streamer':'ST'}[entry['primary_role']]
        trace.stats.location = role_code
        trace.stats.channel = 'GHZ'
        trace.stats.sampling_rate = TARGET_SAMPLING_RATE_HZ
        trace.stats.starttime = VIRTUAL_EPOCH + shot.virtual_shot_number * SHOT_TIME_STRIDE_S + OUTPUT_START_S
        trace.stats.receiver_x_m = receiver_x
        trace.stats.source_x_m = float(shot.source_x_m)
        trace.stats.virtual_shot_number = int(shot.virtual_shot_number)
        trace.stats.receiver_family = entry['primary_family']
        trace.stats.integration_role = entry['primary_role']
        output_stream += trace

        used_ids=[]
        used_roles=[]
        for contribution in entry['contributions']:
            used_ids.append(str(contribution['product_id']))
            used_roles.append(str(contribution['integration_role']))
            contribution_rows.append({
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'output_receiver_x_m': receiver_x,
                **contribution,
                'used_in_output': True,
                'super_stacked': bool(contribution.get('super_stacked', False)),
                'receiver_waveform_corr': contribution.get('receiver_waveform_corr', np.nan),
                'decision': 'base_or_new_receiver' if not contribution.get('super_stacked', False) else 'super_stacked',
            })

        trace_rows.append({
            'virtual_shot_number': int(shot.virtual_shot_number),
            'virtual_shot_id': shot.virtual_shot_id,
            'source_cluster_id': shot.source_cluster_id,
            'source_x_m': float(shot.source_x_m),
            'trace_number_within_shot': trace_number,
            'receiver_family': entry['primary_family'],
            'integration_role': entry['primary_role'],
            'receiver_x_m': receiver_x,
            'station': trace.stats.station,
            'channel': trace.stats.channel,
            'sampling_rate_hz': TARGET_SAMPLING_RATE_HZ,
            'relative_start_s': OUTPUT_START_S,
            'relative_end_s': OUTPUT_END_S,
            'shot_timing_method': shot_timing_method,
            'shot_timing_status': shot_timing_status,
            'estimated_shot_time_from_file_start_s': (
                shot_estimated_t0_s
            ),
            'applied_base_shift_s': (
                shot_applied_base_shift_s
            ),
            'n_samples': trace.stats.npts,
            'n_contributing_products': len(entry['contributions']),
            'total_weight': float(entry['weight']),
            'contributing_product_ids': ' | '.join(used_ids),
            'contributing_roles': ' | '.join(used_roles),
        })

    shot_token = f'{int(shot.virtual_shot_number):04d}_x{float(shot.source_x_m):07.1f}m'
    mseed_path = MSEED_ROOT / f'T1_VIRTUAL_SHOT_{shot_token}_{COMPONENT}.mseed'
    if len(output_stream):
        output_stream.write(str(mseed_path), format='MSEED', encoding=MSEED_ENCODING)
        all_shot_stream += output_stream
        status = 'written'
    else:
        mseed_path = None
        status = 'no_output_traces'

    shot_rows.append({
        'virtual_shot_number': int(shot.virtual_shot_number),
        'virtual_shot_id': shot.virtual_shot_id,
        'source_cluster_id': shot.source_cluster_id,
        'source_x_m': float(shot.source_x_m),
        'component': COMPONENT,
        'n_planned_products': len(shot_products),
        'n_loaded_candidate_traces': n_loaded_candidate_traces,
        'n_output_traces': len(output_stream),
        'n_product_errors': len(product_errors),
        'product_errors_json': json.dumps(product_errors),
        'relative_start_s': OUTPUT_START_S,
        'relative_end_s': OUTPUT_END_S,
        'shot_roles': ' | '.join(sorted(shot_roles)),
        'nodal_only_shot': nodal_only_shot,
        'has_triggered_reference': has_triggered_reference,
        'shot_timing_method': shot_timing_method,
        'shot_timing_status': shot_timing_status,
        'estimated_shot_time_from_file_start_s': (
            shot_estimated_t0_s
        ),
        'applied_base_shift_s': shot_applied_base_shift_s,
        'mseed_path': str(mseed_path) if mseed_path else None,
        'status': status,
    })
    if shot.virtual_shot_number % 20 == 0:
        print(f'Processed shot {shot.virtual_shot_number}/{len(shots)} at x={shot.source_x_m:.1f} m')

virtual_shot_manifest = pd.DataFrame(shot_rows)
virtual_trace_manifest = pd.DataFrame(trace_rows)
trace_contributions = pd.DataFrame(contribution_rows)
alignment_qc = pd.DataFrame(alignment_rows)
alignment_pair_qc = pd.DataFrame(alignment_pair_rows)
product_errors = pd.DataFrame(product_error_rows)
receiver_geometry_qc = pd.DataFrame(geometry_qc_rows)
shot_time_origin_qc = pd.DataFrame(shot_time_qc_rows)
multiple_event_qc = pd.DataFrame(multiple_event_qc_rows)

print('Written shot gathers:', int(virtual_shot_manifest.status.eq('written').sum()))
print('Output traces:', len(virtual_trace_manifest))

/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_5341/496497975.py:255: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(sampled, axis=0)
/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_5341/496497975.py:255: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(sampled, axis=0)
/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_5341/496497975.py:360: RuntimeWarning: All-NaN slice encountered
  scores = np.nanmedian(values, axis=0)
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influe

Processed shot 20/159 at x=92.5 m


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Tra

Processed shot 40/159 at x=108.0 m


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)
/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_5341/496497975.py:255: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(sampled, axis=0)
/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_5341/496497975.py:360: RuntimeWarning: All-NaN slice encountered
  scores = np.nanmedian(values, axis=0)
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393:

...

/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Tra

Processed shot 140/159 at x=189.0 m


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)
/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:393: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Tra

Written shot gathers: 159
Output traces: 8302


/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_5341/496497975.py:255: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(sampled, axis=0)


## 5. Write all-shot MiniSEED and catalogs

In [5]:
ALL_SHOTS_MSEED = OUT_ROOT / f'T1_virtual_all_shots_{COMPONENT}.mseed'

if WRITE_ALL_SHOTS_MSEED and len(all_shot_stream):
    all_shot_stream.traces.sort(key=lambda tr: (
        int(tr.stats.virtual_shot_number), float(tr.stats.receiver_x_m)
    ))
    all_shot_stream.write(str(ALL_SHOTS_MSEED), format='MSEED', encoding=MSEED_ENCODING)
    print('Wrote:', ALL_SHOTS_MSEED)

OUTPUTS = {
    'shots': OUT_ROOT / '99_virtual_T1_shot_manifest.csv',
    'traces': OUT_ROOT / '99_virtual_T1_trace_manifest.csv',
    'contributions': OUT_ROOT / '99_virtual_T1_trace_contributions.csv',
    'alignment_qc': OUT_ROOT / '99_virtual_T1_alignment_qc.csv',
    'alignment_pair_qc': OUT_ROOT / '99_virtual_T1_alignment_pair_qc.csv',
    'product_errors': OUT_ROOT / '99_virtual_T1_product_errors.csv',
    'receiver_geometry_qc': OUT_ROOT / '99_virtual_T1_receiver_geometry_qc.csv',
    'shot_time_origin_qc': OUT_ROOT / '99_virtual_T1_shot_time_origin_qc.csv',
    'multiple_event_qc': OUT_ROOT / '99_virtual_T1_multiple_event_qc.csv',
    'summary': OUT_ROOT / '99_virtual_T1_build_summary.csv',
}

virtual_shot_manifest.to_csv(OUTPUTS['shots'], index=False)
virtual_trace_manifest.to_csv(OUTPUTS['traces'], index=False)
trace_contributions.to_csv(OUTPUTS['contributions'], index=False)
alignment_qc.to_csv(OUTPUTS['alignment_qc'], index=False)
alignment_pair_qc.to_csv(OUTPUTS['alignment_pair_qc'], index=False)
product_errors.to_csv(OUTPUTS['product_errors'], index=False)
receiver_geometry_qc.to_csv(OUTPUTS['receiver_geometry_qc'], index=False)
shot_time_origin_qc.to_csv(OUTPUTS['shot_time_origin_qc'], index=False)
multiple_event_qc.to_csv(OUTPUTS['multiple_event_qc'], index=False)

planned_geode_products = int(products.product_kind.eq('geode_raw').sum())
geode_output_traces = int(virtual_trace_manifest.receiver_family.eq('geode').sum()) if len(virtual_trace_manifest) else 0
geode_product_errors = int(product_errors.product_kind.eq('geode_raw').sum()) if len(product_errors) else 0

if REQUIRE_GEODE_OUTPUT_IF_PLANNED and planned_geode_products > 0 and geode_output_traces == 0:
    display(product_errors.loc[product_errors.product_kind.eq('geode_raw')].head(50))
    raise RuntimeError(
        f'Notebook 98 planned {planned_geode_products} raw Geode products, but notebook 99 produced zero Geode output traces. See {OUTPUTS["product_errors"]}'
    )

summary = pd.DataFrame([
    ('virtual_shots_planned', len(shots)),
    ('virtual_shots_written', int(virtual_shot_manifest.status.eq('written').sum())),
    ('virtual_shots_empty', int(virtual_shot_manifest.status.ne('written').sum())),
    ('output_traces', len(virtual_trace_manifest)),
    ('T1_1m_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('T1_1m').sum()) if len(virtual_trace_manifest) else 0),
    ('T1_2m_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('T1_2m').sum()) if len(virtual_trace_manifest) else 0),
    ('nodal_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('nodal').sum()) if len(virtual_trace_manifest) else 0),
    ('T1_Streamer_primary_output_traces', int(virtual_trace_manifest.integration_role.eq('T1_Streamer').sum()) if len(virtual_trace_manifest) else 0),
    ('planned_geode_products', planned_geode_products),
    ('geode_output_traces', geode_output_traces),
    ('geode_product_errors', geode_product_errors),
    ('alignment_products_accepted', int(alignment_qc.alignment_accepted.sum()) if len(alignment_qc) else 0),
    ('alignment_products_rejected', int((~alignment_qc.alignment_accepted).sum()) if len(alignment_qc) else 0),
    ('super_stacked_contributions', int(trace_contributions.super_stacked.fillna(False).sum()) if len(trace_contributions) else 0),
    ('nodal_only_shots', int(virtual_shot_manifest.nodal_only_shot.fillna(False).sum()) if len(virtual_shot_manifest) else 0),
    ('nodal_airwave_fits_accepted', int(
        shot_time_origin_qc.timing_method.astype(str).eq(
            'nodal_airwave_backprojection'
        ).sum()
    ) if len(shot_time_origin_qc) else 0),
    ('nodal_airwave_fits_rejected_or_qc_only', int(
        shot_time_origin_qc.timing_method.astype(str).eq(
            'nodal_airwave_backprojection_qc'
        ).sum()
    ) if len(shot_time_origin_qc) else 0),
    ('manual_shot_time_overrides_applied', int(
        shot_time_origin_qc.timing_method.astype(str).eq(
            'manual_override'
        ).sum()
    ) if len(shot_time_origin_qc) else 0),
    ('multiple_event_or_ambiguous_alignment_flags', len(multiple_event_qc)),
    ('output_start_s', OUTPUT_START_S),
    ('output_end_s', OUTPUT_END_S),
], columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print()
print('Written:')
for name, path in OUTPUTS.items():
    print(f'  {name:20s} {path}')

Wrote: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/T1_virtual_all_shots_Z.mseed


,metric,value
0,virtual_shots_planned,159.00
1,virtual_shots_written,159.00
2,virtual_shots_empty,0.00
3,output_traces,8302.00
4,T1_1m_primary_output_traces,2808.00
5,T1_2m_primary_output_traces,2592.00
6,nodal_primary_output_traces,2902.00
7,T1_Streamer_primary_output_traces,0.00
8,planned_geode_products,155.00
9,geode_output_traces,5400.00



Written:
  shots                /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_shot_manifest.csv
  traces               /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_trace_manifest.csv
  contributions        /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_trace_contributions.csv
  alignment_qc         /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_alignment_qc.csv
  alignment_pair_qc    /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_alignment_pair_qc.csv
  product_errors       /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_product_errors.csv
  receiver_geometry_qc /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_receiver_geometry_qc.csv
  shot_time_origin_qc  /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_shot_time_origin_qc.csv
  multiple_event_qc    /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtu

## 6. Next step

Notebook 100 reads these ordered shot gathers and catalogs and exports:

- sparse all-shot SEG-Y, containing only observed traces;
- regularized all-shot SEG-Y, containing the full master receiver grid with
  missing combinations written as zero-valued traces marked dead;
- optional per-shot SEG-Y files.

The CSV manifests remain authoritative for receiver family and provenance.